In [1]:
#1. 원본 데이터 토큰화

txt = """
1. 풀꽃 / 나태주
자세히 보아야 예쁘다.
오래 보아야 사랑스럽다.
너도 그렇다.

2. 흔들리며 피는 꽃 / 도종환
흔들리지 않고 피는 꽃이 어디 있으랴.
이 세상 그 어떤 아름다운 꽃들도
다 흔들리며 피었나니.

3. 서시 / 윤동주
죽는 날까지 하늘을 우러러
한 점 부끄럼이 없기를,
잎새에 이는 바람에도
나는 괴로워했다.

4. 꽃 / 김춘수
내가 그의 이름을 불러주었을 때
그는 나에게로 와서
꽃이 되었다.

5. 그대가 곁에 있어도 나는 그대가 그립다 / 류시화
우리가 함께 있을 때조차
나는 그대가 그립다.
사랑은 머물지 않고 흘러가며
그리움만 남긴다.

6. 길 / 김소월
인생은 나그네길,
어디서 왔다가 어디로 가는가.
흩날리는 꽃잎처럼
덧없는 것이 사람의 마음이라.

7. 첫눈 / 안도현
첫눈이 내리는 날
우리는 서로의 이름을 불렀다.
하얀 침묵 속에서도
마음은 따뜻했다.

8. 마음의 문 / 이해인
마음의 문을 잠그지 마라.
바람이 스며들고
햇살이 들어와
상처를 치유하리니.

9. 나무 / 정호승
겨울이 와도
나무는 봄을 믿는다.
그 믿음 하나로
눈보라를 견딘다.

10. 별 / 이정하
그대의 눈빛이 별빛이라면
나는 밤마다 하늘을 올려다보리.
그리움이 멀리서도
나를 비추리니.
"""

In [28]:
import tiktoken
import numpy as np
import chromadb

def split_text(full_text, chunksize):
    encoder = tiktoken.encoding_for_model("gpt-5")
    total_encoding = encoder.encode(full_text)
    # print(len(total_encoding))

    total_token_count = len(total_encoding)
    text_list = []
    for i in range(0, total_token_count, chunksize):
        chunk = total_encoding[i: i+chunksize]
        decoded = encoder.decode(chunk)
        text_list.append(decoded)
    return text_list

In [29]:
chunk_list = split_text(txt, 50)
print(len(chunk_list))

9


In [30]:
print(chunk_list[4])

랑은 머물지 않고 흘러가며
그리움만 남긴다.

6. 길 / 김소월
인생은 나그네길,
어디서 왔다가 어디로 가는가.
흩


In [31]:
# 문장을 벡터로 변환

def random_token_embeddings(texts, dim = 786):

    token_vecs=[]
    for token in texts:
        # 토큰 문자열 해시 → 시드
        token_seed = abs(hash(token)) % (2**32)
        # 같은 문장이라면 동일한 벡터값으로, 아니면 걍 랜덤
        rng = np.random.default_rng(token_seed)
        # 우리 벡터는 768 차원
        token_vec = rng.normal(0, 0.1, size=(dim,))
        token_vecs.append(token_vec)
    return np.vstack(token_vecs)

In [32]:
emb = random_token_embeddings(chunk_list)
print(len(emb))

9


In [33]:
ids = ['doc1','doc2','doc3', 'doc4','doc5', 'doc6','doc7', 'doc8','doc9']

In [35]:
client = chromadb.PersistentClient()

In [41]:
collection = client.create_collection(name = "poem1")

In [42]:
collection.add(
    documents = chunk_list,
    ids = ids,
    embeddings = emb.tolist()
)

In [47]:
query = "윤동주 시 알려줘"
q_emb = random_token_embeddings([query])

print(q_emb)

[[-6.58795815e-02 -1.59940549e-01 -2.12038911e-01  9.00411777e-02
   2.31177002e-01 -1.36224326e-01 -1.35364958e-01 -2.44342834e-01
   1.55334621e-02  1.04028155e-01  2.02985862e-01  6.78511975e-02
  -2.50649192e-02  1.20436134e-01  6.75494368e-02  1.88174501e-01
   2.81767474e-02 -2.14000840e-01 -3.87707206e-02  1.51169019e-01
  -1.59446438e-01  1.05364168e-01  6.87014038e-02  9.71676971e-02
  -3.96480489e-02 -1.06561184e-01 -1.18626932e-01 -1.85186103e-02
  -3.15551573e-02  7.34970090e-02 -1.58412535e-01  5.66480416e-02
  -8.64000840e-02  9.23309191e-02  5.02370409e-02  9.01621436e-02
  -6.62759818e-02 -1.64792036e-01 -1.42904657e-02  1.96785454e-01
   1.39549083e-01 -1.01988369e-01 -2.85451907e-02  1.08034402e-01
   1.39326692e-01  8.67264761e-02  1.71297324e-01 -2.77646302e-02
  -1.41206081e-01 -3.78772850e-02  1.46559287e-01 -2.21270171e-02
   2.02912525e-01  1.57303191e-01  2.42221327e-01 -6.94659236e-02
  -1.09608962e-01  3.96459314e-03 -1.44157734e-01 -7.56161502e-02
   1.07586

In [54]:
result = collection.query(query_texts = query, query_embeddings = q_emb.tolist(), n_results = 4)

In [58]:
print(result['documents'][0][0])

날리는 꽃잎처럼
덧없는 것이 사람의 마음이라.

7. 첫눈 / 안도현
첫눈이 내리는 날
우리는 서로의 이름을 불렀다.
하얀 침묵 속
